In [ ]:
# Test dataset: questions, contexts, and ground truth answers
# This dataset contains multiple Q&A pairs to evaluate model performance
# Each item has: a question, a context paragraph, and the expected answer
qa_test_data = [
    {
        'question': 'What is the immune system?',
        'context': 'The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism\'s own healthy tissue.',
        'answer': 'a system of many biological structures and processes within an organism that protects against disease'
    },
    {
        'question': 'What must the immune system detect?',
        'context': 'The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism\'s own healthy tissue.',
        'answer': 'a wide variety of agents, known as pathogens'
    },
    {
        'question': 'What are pathogens?',
        'context': 'The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism\'s own healthy tissue.',
        'answer': 'agents from viruses to parasitic worms'
    }
]

def get_answer(question, context):
    """
    Extract answer from context using BERT Q&A model.
    
    This function:
    1. Prepares the question and context with special tokens
    2. Tokenizes both inputs
    3. Feeds them to the BERT model
    4. Returns the predicted answer span and confidence score
    
    Args:
        question: The question to answer
        context: The paragraph containing the answer
        
    Returns:
        answer: The predicted answer text
        confidence: Combined confidence score (start + end logits)
    """
    # Prepare input: Add [CLS] at start of question, [SEP] at end of both
    question_text = '[CLS] ' + question + '[SEP]'
    context_text = context + '[SEP]'
    
    # Tokenize: Convert text to WordPiece tokens
    question_tokens = tokenizer.tokenize(question_text)
    context_tokens = tokenizer.tokenize(context_text)
    
    # Combine tokens and convert to numerical IDs that BERT understands
    tokens = question_tokens + context_tokens
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    
    # Segment IDs: 0 for question tokens, 1 for context tokens
    # This helps BERT distinguish between question and context
    segment_ids = [0] * len(question_tokens) + [1] * len(context_tokens)
    
    # Convert to PyTorch tensors (add batch dimension with [])
    input_ids = torch.tensor([input_ids])
    segment_ids = torch.tensor([segment_ids])
    
    # Get model output: BERT predicts start and end positions of the answer
    output = model(input_ids=input_ids, token_type_ids=segment_ids)
    
    # Get indices with highest scores: these mark answer boundaries
    start_index = torch.argmax(output.start_logits)
    end_index = torch.argmax(output.end_logits)
    
    # Extract answer: Join tokens between start and end indices
    answer = ' '.join(tokens[start_index:end_index+1])
    
    # Calculate confidence: Higher scores = more confident prediction
    start_score = torch.max(output.start_logits).item()
    end_score = torch.max(output.end_logits).item()
    confidence = start_score + end_score
    
    return answer, confidence

# Evaluate the Q&A model on test dataset
print("Evaluating Q&A Model:\n")
print("="*80)

# Store results for each test case
results = []

# Iterate through each question-answer pair in test dataset
for i, item in enumerate(qa_test_data, 1):
    # Get model prediction and confidence score
    predicted_answer, confidence = get_answer(item['question'], item['context'])
    
    # Calculate metrics: Exact Match and F1 score
    em = calculate_exact_match(predicted_answer, item['answer'])
    f1 = calculate_f1(predicted_answer, item['answer'])
    
    # Save results for averaging later
    results.append({
        'em': em,
        'f1': f1,
        'confidence': confidence
    })
    
    # Display results for this example
    print(f"\nExample {i}:")
    print(f"Question: {item['question']}")
    print(f"Predicted: {predicted_answer}")
    print(f"Truth: {item['answer']}")
    print(f"EM: {em}  |  F1: {f1:.3f}  |  Confidence: {confidence:.2f}")

print("\n" + "="*80)

# Calculate average metrics across all test examples
# This gives us overall model performance
avg_em = sum(r['em'] for r in results) / len(results)
avg_f1 = sum(r['f1'] for r in results) / len(results)
avg_confidence = sum(r['confidence'] for r in results) / len(results)

print(f"\n📊 Average Performance:")
print(f"  Exact Match: {avg_em:.2%}")
print(f"  F1 Score: {avg_f1:.3f}")
print(f"  Avg Confidence: {avg_confidence:.2f}")

print("\n" + "="*80)

### Evaluate on Multiple Examples

Now let's test our Q&A model on several question-context pairs and calculate average metrics.

In [ ]:
def normalize_answer(text):
    """
    Normalize answer text for fair comparison between prediction and ground truth.
    
    Normalization steps:
    - Convert to lowercase (so "System" matches "system")
    - Remove punctuation (so "disease." matches "disease")
    - Remove articles (a, an, the) as they don't affect meaning
    - Remove extra whitespace
    
    This allows us to focus on content similarity rather than formatting differences.
    """
    import re
    import string
    
    # Convert to lowercase for case-insensitive comparison
    text = text.lower()
    
    # Remove all punctuation marks (replace with space to avoid joining words)
    text = ''.join(ch if ch not in string.punctuation else ' ' for ch in text)
    
    # Remove common articles that don't affect answer meaning
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    
    # Collapse multiple spaces into single space and trim
    text = ' '.join(text.split())
    
    return text

def calculate_exact_match(prediction, ground_truth):
    """
    Calculate Exact Match (EM) score.
    
    EM is a strict metric: the prediction must match the ground truth exactly
    after normalization. This is the primary metric for Q&A systems.
    
    Returns:
        1 if normalized strings match exactly
        0 otherwise (no partial credit)
    """
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def calculate_f1(prediction, ground_truth):
    """
    Calculate F1 score based on token-level overlap.
    
    F1 is more lenient than Exact Match - it gives partial credit for
    answers that share some words with the ground truth.
    
    F1 = 2 * (precision * recall) / (precision + recall)
    
    Where:
    - Precision = What fraction of predicted words are correct?
    - Recall = What fraction of ground truth words were predicted?
    
    Example:
    - Prediction: "biological system structures"
    - Truth: "biological structures and processes"
    - Common: {"biological", "structures"}
    - Precision: 2/3, Recall: 2/4, F1: 0.571
    """
    # Tokenize normalized answers into individual words
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    
    # Handle empty predictions or ground truths
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)
    
    # Find common tokens using set intersection
    common_tokens = set(pred_tokens) & set(truth_tokens)
    
    # If no overlap, F1 is 0
    if len(common_tokens) == 0:
        return 0
    
    # Calculate precision: correct tokens / predicted tokens
    precision = len(common_tokens) / len(pred_tokens)
    
    # Calculate recall: correct tokens / ground truth tokens
    recall = len(common_tokens) / len(truth_tokens)
    
    # Calculate F1 score: harmonic mean of precision and recall
    f1 = 2 * (precision * recall) / (precision + recall)
    
    return f1

# Test the metrics with example cases to understand how they work
print("Testing Metrics:\n")
print("="*80)

# Test cases showing different levels of match quality
test_cases = [
    ("a system of many biological structures", "a system of many biological structures", "Perfect match"),
    ("system of biological structures", "a system of many biological structures", "Missing words"),
    ("the biological system", "a system of many biological structures", "Partial match"),
    ("completely wrong answer", "a system of many biological structures", "No match"),
]

for pred, truth, description in test_cases:
    # Calculate both metrics
    em = calculate_exact_match(pred, truth)
    f1 = calculate_f1(pred, truth)
    
    print(f"\n{description}:")
    print(f"  Predicted: '{pred}'")
    print(f"  Truth: '{truth}'")
    print(f"  EM: {em}  |  F1: {f1:.3f}")

print("\n" + "="*80)

# Q&A with finetuned BERT

In this section, let's learn how to perform question answering with a finetuned Q&A BERT. First, let us import the necessary modules:

In [23]:
%%capture
!pip install transformers==3.5.1

In [ ]:
# Import necessary libraries
import torch  # PyTorch for tensor operations
from transformers import BertForQuestionAnswering, BertTokenizer  # Hugging Face BERT models


Now, we download and load the model. We use the bert-large-uncased-whole-word-masking-finetuned-squad model which is finetuned on the SQUAD (Stanford question answering dataset).


In [ ]:
# Load pre-trained BERT model fine-tuned on SQuAD dataset
# This model is specifically trained for extractive question answering
# - bert-large: Larger model with 24 layers (better accuracy, slower)
# - uncased: Not case-sensitive (treats "Hello" and "hello" the same)
# - whole-word-masking: Better pre-training technique
# - finetuned-squad: Trained on Stanford Question Answering Dataset
model = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')


Next, we download and load the tokenizer which is used for pretraining the bert-large-uncased-whole-word-masking-finetuned-squad model:


In [ ]:
# Load the tokenizer that matches the pre-trained model
# The tokenizer converts text to tokens (subwords) that BERT can understand
# It must match the model's vocabulary and special tokens
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')


Now that we downloaded the model and tokenizer, let's preprocess the input.

## Preprocessing the input
First, we define the input to the BERT which is question and paragraph text:


In [ ]:
# Define the question we want to answer
question = "What is the immune system?"

# Define the context paragraph that contains the answer
# BERT will extract a span from this paragraph as the answer
paragraph = "The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism's own healthy tissue."

Add [CLS] token to the beginning of the question and [SEP] token at the end of both the question and paragraph:

In [ ]:
# Add special BERT tokens:
# [CLS] = Classification token at the start (required by BERT)
# [SEP] = Separator token to mark end of segments
question = '[CLS] ' + question + '[SEP]'
paragraph = paragraph + '[SEP]'


Now, tokenize the question and paragraph:


In [ ]:
# Tokenize: Convert text into WordPiece tokens
# Example: "immune" might become ["im", "##mune"]
# The ## prefix indicates a continuation of the previous word
question_tokens = tokenizer.tokenize(question)
paragraph_tokens = tokenizer.tokenize(paragraph)



Combine the question and paragraph tokens and convert them to input_ids:

In [ ]:
# Combine question and paragraph tokens into single sequence
# BERT processes both together: [CLS] question [SEP] paragraph [SEP]
tokens = question_tokens + paragraph_tokens

# Convert tokens to numerical IDs using the vocabulary
# Each token maps to a unique integer ID (e.g., "the" -> 1996)
input_ids = tokenizer.convert_tokens_to_ids(tokens)



Next, we define the segment_ids. The segment_ids will be 0 for all the tokens of question and it will be 1 for all the tokens of the paragraph:


In [ ]:
# Create segment IDs (also called token_type_ids)
# These tell BERT which tokens belong to the question (0) vs. paragraph (1)
# Example: [0, 0, 0, 0, 1, 1, 1, 1, 1, ...]
#          [CLS] What is ... [SEP] The immune system ...
segment_ids = [0] * len(question_tokens)  # All question tokens get 0
segment_ids += [1] * len(paragraph_tokens)  # All paragraph tokens get 1


Now we convert the input_ids and segment_ids to tensor:

In [ ]:
# Convert to PyTorch tensors
# The [] adds a batch dimension (batch_size=1) since we're processing one example
# Shape: (1, sequence_length)
input_ids = torch.tensor([input_ids])
segment_ids = torch.tensor([segment_ids])



Now that we processed the input. Let's feed them to the model and get the result.

## Getting the answer
We feed the input_ids and segment_ids to the model which return the start score and end score for all of the tokens:


In [ ]:
# Feed inputs to the BERT Q&A model
# Model outputs:
# - start_logits: scores for each token being the START of the answer
# - end_logits: scores for each token being the END of the answer
output = model(input_ids=input_ids, token_type_ids = segment_ids)

In [ ]:
# Extract the start and end scores (logits) for each token position
# These are raw scores before softmax - higher score = more likely to be answer boundary
start_scores = output.start_logits
end_scores = output.end_logits


Now, we select the start_index which is the index of the token which has a maximum start score and end_index which is the index of the token which has a maximum end score:


In [ ]:
# Find the token indices with the highest scores
# argmax returns the position of the maximum value
# start_index: where the answer begins
# end_index: where the answer ends
start_index = torch.argmax(start_scores)
end_index = torch.argmax(end_scores)


That's it! Now, we print the text span between the start and end index as our answer:

In [ ]:
# Extract and print the answer: join all tokens between start and end indices
# +1 because Python slicing is exclusive at the end
# This reconstructs the answer text from the token sequence
print(' '.join(tokens[start_index:end_index+1]))

---

## Evaluating Q&A Performance with Metrics

The example above shows how to get a single answer, but how do we measure if our model is performing well? Let's look at key metrics for Question Answering systems.

### Understanding Q&A Metrics

For extractive Question Answering (like BERT Q&A), we use two main metrics:

1. **Exact Match (EM)**: The predicted answer must match the ground truth exactly (after normalization)
   - Binary: either 1 (perfect match) or 0 (no match)
   - Strict but clear measure of success

2. **F1 Score**: Token-level overlap between prediction and ground truth
   - Measures partial credit for partially correct answers
   - More lenient than EM

Let's implement these metrics and test them on multiple examples.